# P3. 상권 분석

소상공인 상가(상권)정보의 서울 음식점 14만 곳과 서울 행정동 생활인구 한 달치를 ERD 로 설계해 DuckDB 에 정규화된 테이블로 적재하고, SQL 로 「생활인구 1만 명당 카페 수」가 높은 동네를 찾습니다. 두 출처의 행정동 코드가 어긋나는 곳을 외래 키 제약으로 잡아냅니다.

- 단계: 개별 프로젝트 · 정형 · 예상 2~3일
- 실행 환경: Google Colab(브라우저만 있으면 됩니다)
- 데이터: 공공데이터포털 — 소상공인시장진흥공단_상가(상권)정보(15083033) 2026-06-30 기준 서울 파일 중 음식 업종, 이용허락 제한 없음
- 데이터: 서울 열린데이터광장 — 행정동 단위 서울 생활인구(내국인)(OA-14991) 2026년 6월분, 공공누리 1유형(출처표시)

**빈칸 노트북** — 코드 속 `___①___` 자리 13곳을 채웁니다. 줄 끝 주석이 힌트입니다.
막히면 [따라하기 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/P3/P3_guide.ipynb)의 같은 단계를 보세요. 정답은 맨 아래에 있습니다.

「카페를 내려면 어디가 좋을까?」 하는 질문에는 숫자 두 개가 필요합니다. 그 동네에 **카페가 몇 곳 있는지**, 그리고 그 동네에 **사람이 얼마나 머무는지**입니다.
첫째는 소상공인시장진흥공단의 상가(상권)정보, 둘째는 서울시의 생활인구(통신 데이터로 추정한, 그 시각 그 동네에 있던 사람 수)에 있습니다.

두 자료는 만든 기관도, 갱신 주기도, 쓰는 코드표도 다릅니다. 그래서 이 프로젝트의 중심은 SQL 한 줄이 아니라 **두 자료를 한 데이터베이스에 바르게 담는 설계**입니다. 2주차 10장의 ERD 와 정규화를 실제 공공데이터에 적용합니다.

```text
상가정보 CSV ─┐                          ┌→ 차원: 자치구 · 행정동 · 업종(대 · 중 · 소)
              ├→ ERD 설계 → DuckDB DDL ──┤
생활인구 CSV ─┘   (PK · FK)               └→ 사실: 점포 · 생활인구  →  SQL 분석 · 그래프
```

다 끝내면 다음을 말할 수 있습니다.

- 서울에서 생활인구 1만 명당 카페가 가장 많은 행정동과, 카페 수 자체가 가장 많은 행정동이 왜 다른지
- 업종 코드를 왜 대 · 중 · 소 세 테이블로 나누었는지(정규화)
- 두 출처의 행정동 코드가 어디서 어긋나는지, 외래 키가 그것을 어떻게 잡아냈는지

## 1단계 · 준비 — 라이브러리와 데이터

Colab 에는 pandas 와 DuckDB 가 이미 깔려 있습니다. 데이터는 공개 자료실에서 받습니다.

- **상가(상권)정보** — 공공데이터포털 「소상공인시장진흥공단_상가(상권)정보」 2026-06-30 기준 파일(전국, 지역별 CSV 16개, UTF-8)에서 서울 파일을 골라, 상권업종 대분류가 「음식」인 점포 141,126곳만 남겼습니다. 열은 39개 중 분석에 쓸 15개만 남겼고 값은 원본 그대로입니다. 이용허락 범위는 「제한 없음」입니다.
- **생활인구** — 서울 열린데이터광장 「행정동 단위 서울 생활인구(내국인)」 2026년 6월 파일입니다. 원본의 32개 열 중 앞 4개(날짜 · 시간대 · 행정동코드 · 총생활인구수)만 남겼고, 행 모양(UTF-8 BOM, 데이터 행 끝의 쉼표)은 원본 그대로 두었습니다. 공공누리 1유형(출처표시)입니다.

In [ ]:
import os, shutil, urllib.request
import pandas as pd
import duckdb

DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")          # 오프라인 실습용 — 자료 폴더가 내 컴퓨터에 있을 때
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

store_path = fetch("seoul_store_food_202606.csv.gz")
pop_path = fetch("seoul_living_pop_dong_202606.csv.gz")
for p in (store_path, pop_path):
    print(p, round(os.path.getsize(p) / 1e6, 1), "MB")

> **확인 포인트** — `seoul_store_food_202606.csv.gz 5.8 MB` · `seoul_living_pop_dong_202606.csv.gz 2.6 MB`.

> **Aebon's Tip** — 서울 생활인구는 2026년 7월분을 끝으로 행정동 단위 생산이 끝나고, 국가표준격자(250m) 단위로 바뀐다고 데이터셋 화면에 안내되어 있습니다.
> 앞으로의 자료로 같은 분석을 하려면 격자를 행정동에 겹쳐 합치는 공간 조인이 한 단계 더 필요합니다.

## 2단계 · 읽기 — 행 끝 쉼표에 속아 보기

상가정보는 UTF-8 이라 그대로 읽힙니다. 점포 번호(`상가업소번호`)와 코드 열의 앞자리 0이 사라지지 않도록 모든 열을 글자로 읽습니다(M1 2단계와 같은 이유).

In [ ]:
store_raw = pd.read_csv(store_path, dtype=str)
print(store_raw.shape)
store_raw.head(3)

> **확인 포인트** — `(141126, 15)`.

In [ ]:
assert store_raw.shape == (141126, 15)

생활인구를 같은 방법으로 읽어 봅니다.

In [ ]:
bad = pd.read_csv(pop_path, dtype=str)
print(bad.shape)
print(bad.head(3))
print("총생활인구수 빈 값:", bad["총생활인구수"].isna().sum())

오류는 나지 않았는데 표가 이상합니다. 날짜가 왼쪽 행 번호 자리로 가고, `기준일ID` 열에 시간대(`00`)가, `총생활인구수` 열에는 빈 값만 들어 있습니다.
원본 파일은 **머리줄은 4칸인데 데이터 줄 끝에 쉼표가 하나 더 붙어 5칸**입니다. pandas 는 칸이 하나 많으면 맨 앞 칸을 행 번호(index)로 써 버립니다. 조용히 틀리는 실패라 더 위험합니다.

```text
"기준일ID","시간대구분","행정동코드","총생활인구수"
"20260601","00","11740685","61126.7274",
```

`index_col=False` 를 주면 「맨 앞 칸을 행 번호로 쓰지 말라」는 뜻이 되어, 남는 마지막 빈칸을 버리고 제대로 읽습니다.

In [ ]:
pop_raw = pd.read_csv(pop_path, dtype=str, index_col=___①___)   # ① 맨 앞 칸을 행 번호로 쓰지 않기
print(pop_raw.shape)
print(pop_raw.head(3))
print("날짜:", pop_raw["기준일ID"].min(), "~", pop_raw["기준일ID"].max(), "· 행정동:", pop_raw["행정동코드"].nunique())

> **확인 포인트** — `(305280, 4)` — 30일 × 24시간 × 행정동 424곳입니다. 이번에는 빈 값이 없습니다.

In [ ]:
assert pop_raw.shape == (305280, 4) and pop_raw["행정동코드"].nunique() == 424
assert pop_raw.isna().sum().sum() == 0 and bad["총생활인구수"].isna().all()

## 3단계 · ERD 설계 — 무엇을 따로 떼어 낼까

상가정보 한 줄에는 점포 정보와 함께 업종 이름 · 행정동 이름 · 구 이름이 반복해서 들어 있습니다. 이대로 한 테이블에 두면 「카페」라는 업종 이름이 2만 번 넘게 저장되고, 이름 하나를 고치려면 2만 줄을 고쳐야 합니다(갱신 이상 — 2주차 10장).

정규화의 출발점은 **함수 종속**입니다. 「A 를 알면 B 가 하나로 정해진다」(A → B)가 성립하면 A 를 키로 하는 테이블로 B 를 떼어 낼 수 있습니다. 데이터에서 직접 확인합니다.

In [ ]:
def fd(df, a, b):
    """a → b 가 성립하면 a 값마다 b 의 종류가 1개입니다."""
    return df.groupby(a)[b].___②___().max() == 1   # ② 서로 다른 값의 개수

checks = [
    ("상권업종소분류코드", "상권업종소분류명"), ("상권업종소분류코드", "상권업종중분류코드"),
    ("상권업종중분류코드", "상권업종중분류명"), ("상권업종중분류코드", "상권업종대분류코드"),
    ("행정동코드", "행정동명"), ("행정동코드", "시군구코드"), ("시군구코드", "시군구명"),
]
for a, b in checks:
    print(f"{a} → {b}:", fd(store_raw, a, b))
print("상가업소번호가 겹치지 않는가:", store_raw["상가업소번호"].is_unique)

> **확인 포인트** — 일곱 줄이 모두 `True`, 상가업소번호도 `True` 입니다.

In [ ]:
assert all(fd(store_raw, a, b) for a, b in checks) and store_raw["상가업소번호"].is_unique

반대 방향은 성립하지 않습니다. 이름이 같은 행정동이 다른 구에 또 있기 때문입니다. 그래서 행정동의 키는 이름이 아니라 **코드**여야 합니다.

In [ ]:
dup = store_raw.groupby("행정동명")["행정동코드"].nunique()
print("이름만으로는 하나로 정해지지 않는 행정동:", dup[dup > 1].index.tolist()[:8])

> **확인 포인트** — `신사동` 은 강남구와 관악구에 하나씩 있습니다.

In [ ]:
assert "신사동" in dup[dup > 1].index

확인한 함수 종속대로 테이블을 나눕니다. 관계는 모두 「하나 대 여럿」입니다 — 구 하나에 행정동 여럿, 행정동 하나에 점포 여럿, 소분류 하나에 점포 여럿.

```text
erDiagram
    SGG            ||--o{ DONG           : "has"
    DONG           ||--o{ STORE          : "located in"
    DONG           ||--o{ LIVING_POP     : "measured in"
    INDUSTRY_LARGE ||--o{ INDUSTRY_MID   : "groups"
    INDUSTRY_MID   ||--o{ INDUSTRY_SMALL : "groups"
    INDUSTRY_SMALL ||--o{ STORE          : "classifies"

    SGG {
        varchar sgg_cd PK "자치구코드 5자리"
        varchar sgg_nm
    }
    DONG {
        varchar dong_cd PK "행정동코드 8자리"
        varchar dong_nm
        varchar sgg_cd FK
    }
    INDUSTRY_LARGE {
        varchar large_cd PK
        varchar large_nm
    }
    INDUSTRY_MID {
        varchar mid_cd PK
        varchar mid_nm
        varchar large_cd FK
    }
    INDUSTRY_SMALL {
        varchar small_cd PK
        varchar small_nm
        varchar mid_cd FK
    }
    STORE {
        varchar store_id PK "상가업소번호"
        varchar store_nm
        varchar branch_nm
        varchar small_cd FK
        varchar dong_cd FK
        double lon
        double lat
    }
    LIVING_POP {
        date base_date PK
        smallint hour PK
        varchar dong_cd PK, FK
        double total_pop
    }
```

이 글을 [mermaid.live](https://mermaid.live) 에 붙이면 그림으로 볼 수 있습니다. 생활인구의 기본 키는 (날짜, 시간대, 행정동) 세 열을 묶은 **복합 키**입니다. 같은 동네 · 같은 시각의 값은 하나뿐이어야 하니까요.

> **Aebon's Tip** — 시군구명 · 행정동명을 점포 테이블에 그대로 두는 「반정규화」도 흔합니다. 조회가 빨라지는 대신 이름이 바뀌면 여러 곳을 고쳐야 합니다.
> 분석용 마트에서는 반정규화, 원천을 담는 테이블에서는 정규화 — 이 프로젝트는 원천부터 바르게 담는 연습입니다.

## 4단계 · DDL 과 차원 테이블 — 키 제약 걸기

ERD 를 그대로 `CREATE TABLE` 로 옮깁니다. DuckDB 는 기본 키(PRIMARY KEY)와 외래 키(REFERENCES)를 실제로 검사합니다. 파일 하나(`shop.duckdb`)에 저장해 두면 노트북을 다시 열어도 남습니다(3주차 14장).

부모 테이블(구 · 대분류)부터 만들어야 자식 테이블의 REFERENCES 가 그 테이블을 찾을 수 있습니다.

In [ ]:
if os.path.exists("shop.duckdb"):
    os.remove("shop.duckdb")                     # 다시 실행해도 처음부터
con = duckdb.connect("shop.duckdb")

con.execute("""
CREATE TABLE sgg (
    sgg_cd   VARCHAR PRIMARY KEY,
    sgg_nm   VARCHAR NOT NULL
);
CREATE TABLE dong (
    dong_cd  VARCHAR PRIMARY KEY,
    dong_nm  VARCHAR NOT NULL,
    sgg_cd   VARCHAR NOT NULL ___③___ sgg(sgg_cd)   -- ③ 외래 키
);
CREATE TABLE industry_large (large_cd VARCHAR PRIMARY KEY, large_nm VARCHAR NOT NULL);
CREATE TABLE industry_mid (
    mid_cd   VARCHAR PRIMARY KEY,
    mid_nm   VARCHAR NOT NULL,
    large_cd VARCHAR NOT NULL REFERENCES industry_large(large_cd)
);
CREATE TABLE industry_small (
    small_cd VARCHAR PRIMARY KEY,
    small_nm VARCHAR NOT NULL,
    mid_cd   VARCHAR NOT NULL REFERENCES industry_mid(mid_cd)
);
CREATE TABLE store (
    store_id  VARCHAR PRIMARY KEY,
    store_nm  VARCHAR NOT NULL,
    branch_nm VARCHAR,
    small_cd  VARCHAR NOT NULL REFERENCES industry_small(small_cd),
    dong_cd   VARCHAR NOT NULL REFERENCES dong(dong_cd),
    lon DOUBLE, lat DOUBLE
);
CREATE TABLE living_pop (
    base_date DATE,
    hour      SMALLINT,
    dong_cd   VARCHAR REFERENCES dong(dong_cd),
    total_pop DOUBLE NOT NULL,
    ___④___ (base_date, hour, dong_cd)   -- ④ 기본 키
);
""")
con.sql("SELECT table_name FROM duckdb_tables() ORDER BY table_name").df()

차원 테이블은 상가정보에서 `drop_duplicates` 로 뽑아 채웁니다. DuckDB 는 파이썬 안의 pandas 표 이름을 그대로 SQL 에서 읽습니다(M1 4단계).

In [ ]:
s = store_raw
sgg_df = s[["시군구코드", "시군구명"]].___⑤___()   # ⑤ 겹치는 행 없애기
dong_df = s[["행정동코드", "행정동명", "시군구코드"]].drop_duplicates()
large_df = s[["상권업종대분류코드", "상권업종대분류명"]].drop_duplicates()
mid_df = s[["상권업종중분류코드", "상권업종중분류명", "상권업종대분류코드"]].drop_duplicates()
small_df = s[["상권업종소분류코드", "상권업종소분류명", "상권업종중분류코드"]].drop_duplicates()

con.execute("INSERT INTO sgg SELECT * FROM sgg_df")
con.execute("INSERT INTO dong SELECT * FROM dong_df")
con.execute("INSERT INTO industry_large SELECT * FROM large_df")
con.execute("INSERT INTO industry_mid SELECT * FROM mid_df")
con.execute("INSERT INTO industry_small SELECT * FROM small_df")

for t in ["sgg", "dong", "industry_large", "industry_mid", "industry_small"]:
    print(t, con.execute(f"SELECT count(*) FROM {t}").fetchall()[0][0])

> **확인 포인트** — 구 25 · 행정동 427 · 대분류 1(음식) · 중분류 10 · 소분류 43.

In [ ]:
counts = {t: con.execute(f"SELECT count(*) FROM {t}").fetchall()[0][0]
          for t in ["sgg", "dong", "industry_large", "industry_mid", "industry_small"]}
assert counts == {"sgg": 25, "dong": 427, "industry_large": 1, "industry_mid": 10, "industry_small": 43}

이제 **키 제약을 일부러 어겨** 봅니다. 이미 있는 구 코드를 이름만 바꿔 한 번 더 넣으면 기본 키가, 없는 구 코드를 가진 행정동을 넣으면 외래 키가 막습니다.

In [ ]:
for sql in [
    "INSERT INTO sgg VALUES ('11680', '강남')",                    # 강남구 코드 중복
    "INSERT INTO dong VALUES ('99999999', '없는동', '99999')",      # 없는 구를 가리킴
    "DROP TABLE sgg",                                               # 자식(dong)이 가리키는 부모를 지우기
]:
    try:
        con.execute(sql)
        print("통과?!", sql)
    except duckdb.Error as e:
        print(type(e).__name__, "—", str(e).splitlines()[0][:100])

> **자주 겪는 오류** — DuckDB 의 외래 키는 MySQL(7장)보다 기능이 적습니다. `ON DELETE CASCADE` 같은 연쇄 동작이 없고, 부모 행을 지우거나 키를 바꾸려면 자식 행부터 정리해야 합니다.

## 5단계 · 사실 테이블 적재 — 외래 키가 잡아낸 코드 불일치

점포를 넣습니다. 차원 테이블을 같은 파일에서 뽑았으니 모든 외래 키가 맞아야 합니다.

In [ ]:
store_df = s[["상가업소번호", "상호명", "지점명", "상권업종소분류코드", "행정동코드", "경도", "위도"]].copy()
store_df["경도"] = store_df["경도"].astype(float)
store_df["위도"] = store_df["위도"].astype(float)
con.execute("INSERT INTO store SELECT * FROM store_df")
print("점포:", con.execute("SELECT count(*) FROM store").fetchall()[0][0])

같은 점포를 두 번 넣으면 기본 키가 막습니다. 적재 스크립트를 실수로 두 번 돌린 상황입니다.

In [ ]:
try:
    con.execute("INSERT INTO store SELECT * FROM store_df LIMIT 5")
except duckdb.___⑥___ as e:   # ⑥ 제약 위반 오류
    print("두 번째 적재 거절:", str(e).splitlines()[0][:90])

다음은 생활인구입니다. 날짜와 시간대를 알맞은 형으로 바꾸어 넣습니다.

In [ ]:
pop_df = pd.DataFrame({
    "base_date": pd.to_datetime(pop_raw["기준일ID"], format="%Y%m%d").dt.date,
    "hour": pop_raw["시간대구분"].astype(int),
    "dong_cd": pop_raw["행정동코드"],
    "total_pop": pop_raw["총생활인구수"].astype(float),
})
try:
    con.execute("INSERT INTO living_pop SELECT * FROM pop_df")
except duckdb.ConstraintException as e:
    print("적재 실패:", str(e).splitlines()[0][:110])
print("들어간 행:", con.execute("SELECT count(*) FROM living_pop").fetchall()[0][0])

실패했고, 한 행도 들어가지 않았습니다. INSERT 한 문장은 하나의 트랜잭션이라 **전부 들어가거나 하나도 안 들어갑니다**.
생활인구에 있는 행정동 코드 가운데 상가정보(우리의 `dong` 테이블)에 없는 것이 있다는 뜻입니다. 어느 코드인지 양쪽 방향으로 찾습니다 — `NOT IN` 으로 「한쪽에만 있는 것」을 고르는 안티 조인입니다.

In [ ]:
pop_only = con.sql("""
    SELECT DISTINCT dong_cd FROM pop_df
    WHERE dong_cd ___⑦___ (SELECT dong_cd FROM dong)   -- ⑦ 저쪽에 없는 것만
    ORDER BY dong_cd
""").df()
store_only = con.sql("""
    SELECT d.dong_cd, d.dong_nm, g.sgg_nm, count(st.store_id) AS stores
    FROM dong d JOIN sgg g USING (sgg_cd) LEFT JOIN store st USING (dong_cd)
    WHERE d.dong_cd NOT IN (SELECT DISTINCT dong_cd FROM pop_df)
    GROUP BY ALL ORDER BY d.dong_cd
""").df()
print("생활인구에만 있는 코드:", pop_only["dong_cd"].tolist())
store_only

> **확인 포인트** — 생활인구에만 있는 코드 **9개**, 상가정보에만 있는 행정동 **12곳**입니다.
> 강동구를 보면 상가정보에는 상일1동(11740525) · 상일2동(11740526)이 있는데, 생활인구에는 11740520 하나만 있습니다. 동이 나뉘기 전의 코드로 보입니다.
> 강북구 번1~3동 · 수유1~3동도 코드 끝자리가 조금씩 다릅니다. **행정동은 분동 · 통합으로 코드가 바뀌고, 기관마다 코드표를 갱신한 시점이 다릅니다.**

In [ ]:
assert len(pop_only) == 9 and len(store_only) == 12
assert "11740520" in set(pop_only["dong_cd"]) and {"11740525", "11740526"} <= set(store_only["dong_cd"])

코드가 바뀐 곳을 이어 주려면 행정안전부 행정동 코드 변경 내역으로 대응표를 만들어야 합니다. 여기서는 짐작으로 잇지 않고, **양쪽에 다 있는 행정동만** 분석에 씁니다. 빠지는 생활인구 행은 9개 코드 × 720시간입니다.

In [ ]:
pop_ok = pop_df[pop_df["dong_cd"].isin(set(con.sql("SELECT dong_cd FROM dong").df()["dong_cd"]))]
con.execute("INSERT INTO living_pop SELECT * FROM pop_ok")
n_pop = con.execute("SELECT count(*) FROM living_pop").fetchall()[0][0]
print("생활인구:", n_pop, "· 빠진 행:", len(pop_df) - n_pop)

> **확인 포인트** — 생활인구 **298,800행**(415개 동 × 720시간)이 들어가고, 6,480행(9 × 720)이 빠집니다.

In [ ]:
assert n_pop == 298800 and len(pop_df) - n_pop == 6480

> **Aebon's Tip** — 이렇게 빠진 행정동은 분석 결과 옆에 꼭 적어 두세요. 「강북구 카페 밀도가 낮다」가 사실인지, 강북구 동 여섯 곳이 빠져서인지 읽는 사람이 구별할 수 있어야 합니다.

## 6단계 · SQL 분석 — 생활인구 1만 명당 카페 수

먼저 카페가 어떤 업종 코드인지 업종 테이블에서 찾습니다. 카페는 중분류 「비알코올」(음료) 아래에 있습니다.

In [ ]:
q = """
    SELECT s.small_cd, s.small_nm, m.mid_nm, count(st.store_id) AS stores
    FROM industry_small s JOIN industry_mid m USING (mid_cd) LEFT JOIN store st USING (small_cd)
    WHERE m.mid_nm = '비알코올'
    GROUP BY ALL
"""
print("찾은 행:", len(con.sql(q).df()))

한 행도 나오지 않습니다. 이름이 정말 그런지 대괄호로 감싸 눈으로 확인해 봅니다.

In [ ]:
con.sql("SELECT mid_cd, '[' || mid_nm || ']' AS shown, ___⑧___(mid_nm) AS len FROM industry_mid ORDER BY mid_cd").df()   # ⑧ 글자 수

원본 자료의 `비알코올` 뒤에 **공백이 하나** 붙어 있습니다(`[비알코올 ]`, 5글자). 눈으로는 안 보이는 이런 차이가 `=` 비교를 조용히 실패시킵니다.
업종 이름을 차원 테이블 한 곳에만 두었으니, 고칠 곳도 한 곳입니다 — 정규화의 덕을 바로 봅니다.

In [ ]:
con.execute("UPDATE industry_mid SET mid_nm = ___⑨___(mid_nm)")   # ⑨ 앞뒤 공백 지우기
con.sql(q).df()

> **확인 포인트** — 중분류 「비알코올」 아래 소분류는 `I21201 카페` 하나이고, 서울에 **22,739곳**입니다.

In [ ]:
assert con.sql(q).df()["stores"].tolist() == [22739]
cafe_total = con.execute("SELECT count(*) FROM store WHERE small_cd = 'I21201'").fetchall()[0][0]
assert cafe_total == 22739

> **Aebon's Tip** — 정석은 4단계에서 차원 테이블에 넣을 때 `.str.strip()` 으로 모든 이름의 앞뒤 공백을 지우는 것입니다. 적재 스크립트를 다시 쓸 때 그렇게 고쳐 두세요.

행정동마다 두 숫자를 만들어 붙입니다. 생활인구는 한 달 720개 시각의 **평균**(그 동네에 평소 머무는 사람 수)을 씁니다.
CTE(`WITH`)로 단계를 나누면 SQL 을 위에서 아래로 읽을 수 있습니다(2주차 8장).

In [ ]:
density = con.sql("""
    WITH cafe AS (
        SELECT dong_cd, count(*) AS cafes
        FROM store WHERE small_cd = 'I21201'
        GROUP BY dong_cd
    ), pop AS (
        SELECT dong_cd, ___⑩___(total_pop) AS avg_pop   -- ⑩ 720개 시각의 평균
        FROM living_pop
        GROUP BY dong_cd
    )
    SELECT g.sgg_nm, d.dong_nm, d.dong_cd, c.cafes, round(p.avg_pop) AS avg_pop,
           round(c.cafes / p.avg_pop * 10000, 1) AS cafes_per_10k
    FROM pop p
    JOIN dong d USING (dong_cd)
    JOIN sgg g USING (sgg_cd)
    LEFT JOIN cafe c USING (dong_cd)
    ORDER BY cafes_per_10k DESC NULLS LAST
""").df()
print(len(density), "개 동")
density.head(10)

> **확인 포인트** — 415개 동. 1위는 **마포구 연남동 — 생활인구 1만 명당 카페 123.0곳**(카페 210곳, 평균 생활인구 17,074명)입니다.
> 2 · 3위는 종로구 삼청동(98.8) · 가회동(96.6) — 사람은 적은데 카페가 모인 관광 골목입니다.

In [ ]:
assert len(density) == 415
assert density.iloc[0]["dong_nm"] == "연남동" and density.iloc[0]["cafes"] == 210 and density.iloc[0]["cafes_per_10k"] == 123.0

카페 **수** 순위와 나란히 놓아 봅니다.

In [ ]:
by_count = density.sort_values("cafes", ascending=False).head(5)[["sgg_nm", "dong_nm", "cafes", "avg_pop", "cafes_per_10k"]]
by_count

> **확인 포인트** — 카페가 가장 많은 곳은 **마포구 서교동 535곳**, 다음은 강남구 역삼1동 463곳입니다.
> 역삼1동 · 여의동은 카페가 많아도 생활인구가 10만 명 가까이 되어 1만 명당으로는 47.5 · 32.0 에 그칩니다. **「많다」와 「촘촘하다」는 다른 질문입니다.**

In [ ]:
assert by_count.iloc[0]["dong_nm"] == "서교동" and by_count.iloc[0]["cafes"] == 535
assert by_count.iloc[1]["dong_nm"] == "역삼1동" and by_count.iloc[1]["cafes"] == 463

이번에는 구마다 밀도 1위 동을 뽑습니다. 「그룹마다 1등」은 윈도 함수 `row_number()` 로 풉니다(2주차 8장, M1 4단계).

In [ ]:
top_in_gu = con.sql("""
    SELECT sgg_nm, dong_nm, cafes, avg_pop, cafes_per_10k FROM (
        SELECT *, row_number() OVER (PARTITION BY ___⑪___ ORDER BY cafes_per_10k DESC) AS r   -- ⑪ 구마다 따로 순위
        FROM density
        WHERE avg_pop >= 10000
    )
    WHERE r = 1
    ORDER BY cafes_per_10k DESC
""").df()
top_in_gu.head(8)

> **확인 포인트** — 25개 구마다 한 줄씩입니다. 생활인구 1만 명 이상만 남기니 삼청동 · 가회동이 빠지고, 종로구 1위는 사직동(72.4)이 됩니다.

In [ ]:
assert len(top_in_gu) == 25
assert top_in_gu.set_index("sgg_nm").loc["종로구", "dong_nm"] == "사직동"

> **자주 겪는 오류** — 생활인구가 아주 적은 동은 카페 몇 곳만 있어도 밀도가 크게 튑니다. 위 SQL 의 `avg_pop >= 10000` 처럼 하한을 두고, 그 기준을 결과와 함께 적으세요.

## 7단계 · 낮과 밤 — 시간대를 나눠 보기

카페 손님은 낮에 옵니다. 같은 동네라도 낮(12~17시)과 밤(0~5시)의 생활인구는 크게 다릅니다. 오피스 상권은 낮에 몇 배로 붐비고, 주거지는 반대입니다.

In [ ]:
day_night = con.sql("""
    SELECT d.dong_nm, g.sgg_nm,
           avg(total_pop) FILTER (WHERE hour ___⑫___) AS day_pop,   -- ⑫ 낮 시간대
           avg(total_pop) FILTER (WHERE hour BETWEEN 0 AND 5)  AS night_pop
    FROM living_pop JOIN dong d USING (dong_cd) JOIN sgg g USING (sgg_cd)
    GROUP BY ALL
""").df()
day_night["day_night_ratio"] = (day_night["day_pop"] / day_night["night_pop"]).round(2)
day_night.sort_values("day_night_ratio", ascending=False).head(5)

> **확인 포인트** — 낮/밤 비율 1위는 **중구 소공동 8.78배**, 이어서 회현동 · 명동입니다 — 밤에는 비고 낮에 붐비는 도심 업무 · 관광지입니다.

In [ ]:
top_dn = day_night.sort_values("day_night_ratio", ascending=False).iloc[0]
assert top_dn["dong_nm"] == "소공동" and top_dn["day_night_ratio"] == 8.78

밤에 비해 낮 인구가 많은 동네일수록 카페가 촘촘한지, 그림으로 확인합니다.

In [ ]:
import matplotlib.pyplot as plt

dn = day_night.merge(density, on=["sgg_nm", "dong_nm"])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.scatter(dn["avg_pop"], dn["cafes"], s=10, alpha=.5)
ax1.set_xscale("log")
ax1.set_xlabel("average living population (log)")
ax1.set_ylabel("cafes")
ax1.set_title("Cafes vs living population, 415 dongs")

ax2.scatter(dn["day_night_ratio"], dn["cafes_per_10k"], s=10, alpha=.5, color="tab:orange")
ax2.set_xscale("log")
ax2.set_xlabel("day / night population ratio (log)")
ax2.set_ylabel("cafes per 10k people")
ax2.set_title("Office-like dongs have denser cafes?")
for a in (ax1, ax2):
    a.grid(alpha=.3)
plt.tight_layout()
plt.show()

corr = dn[["day_night_ratio", "cafes_per_10k"]].corr(method="spearman").iloc[0, 1]
print("순위 상관(Spearman):", round(corr, 2))

> **확인 포인트** — 왼쪽 그림은 사람이 많을수록 카페도 많은 오른쪽 위 방향, 오른쪽 그림도 낮/밤 비율이 클수록 밀도가 높은 방향입니다. 순위 상관은 **0.52** 입니다.
> 상관은 원인이 아닙니다. 「낮에 사람이 몰리는 곳에 카페가 따라 생긴다」는 가설로 적고, 개업 · 폐업 시점 자료로 확인해야 합니다.

In [ ]:
assert round(corr, 2) == 0.52

## 8단계 · 저장과 정리

결과 표는 Parquet 으로 내보냅니다. DuckDB 는 `COPY ... TO` 한 줄로 씁니다(3주차 14장).

In [ ]:
con.execute("COPY (SELECT * FROM density) TO 'data/cafe_density_202606.parquet' (___⑬___)")   # ⑬ 파일 형식
back = duckdb.sql("SELECT count(*) FROM 'data/cafe_density_202606.parquet'").fetchall()[0][0]
print("다시 읽은 행:", back)
assert back == 415

테이블 크기도 봅니다. 정규화 덕분에 업종 이름은 43줄에만 저장됩니다.

In [ ]:
con.sql("""
    SELECT table_name, estimated_size AS rows, column_count
    FROM duckdb_tables() ORDER BY estimated_size DESC
""").df()

In [ ]:
con.close()
print("shop.duckdb", round(os.path.getsize("shop.duckdb") / 1e6, 1), "MB")

## 9단계 · 더 해 보기

- 행정안전부 「행정기관(행정동) 및 관할구역 변경 내역」을 찾아, 5단계에서 빠진 9개 코드와 12개 행정동의 대응표(`dong_code_map` 테이블)를 만들어 보세요. 나뉜 동은 생활인구를 어떻게 나눌지(점포 수 비율? 면적 비율?) 정하고 이유를 적습니다.
- 카페 대신 `I21` 아래 다른 소분류(제과점 · 아이스크림 등)나 편의점(소매 대분류 — 원본 파일 전체를 받아야 합니다)으로 같은 분석을 해 보세요. 업종 테이블을 따로 둔 덕분에 SQL 의 코드 하나만 바꾸면 됩니다.
- 점포의 경도 · 위도로 행정동 경계를 다시 확인해 보세요. 상가정보의 행정동 코드가 좌표와 맞지 않는 점포가 있다면, 그것도 정제 대상입니다.
- 같은 ERD 를 MySQL(7장)로 옮기고 `ON DELETE` 동작을 붙여 보세요. DuckDB 에서 막혔던 「부모 행 지우기」가 어떻게 달라지나요?

## 정답

① `False` — 맨 앞 칸을 행 번호로 쓰지 않기
② `nunique` — 서로 다른 값의 개수
③ `REFERENCES` — 외래 키
④ `PRIMARY KEY` — 기본 키
⑤ `drop_duplicates` — 겹치는 행 없애기
⑥ `ConstraintException` — 제약 위반 오류
⑦ `NOT IN` — 저쪽에 없는 것만
⑧ `length` — 글자 수
⑨ `trim` — 앞뒤 공백 지우기
⑩ `avg` — 720개 시각의 평균
⑪ `sgg_nm` — 구마다 따로 순위
⑫ `BETWEEN 12 AND 17` — 낮 시간대
⑬ `FORMAT parquet` — 파일 형식